# Fisheye Tomography Transform

In [ ]:
import os

import cv2

import mon
from fisheye import FisheyeTomographyTransform

# Arguments
# image_stem = "0000135_01327_d_0000151"
image_stem = "0000305_00001_d_0000213"
f          = 150
# f          = 300

# Directories and files
current_dir     = mon.Path(os.getcwd())
root_dir        = current_dir.parents[0]
data_dir        = root_dir / "data" / "ftt"
image_file      = data_dir / f"{image_stem}.jpg"
label_file      = data_dir / f"{image_stem}.txt"
classes_file    = data_dir / f"classes.yaml"
fft_image_file  = data_dir / f"{image_stem}_ftt_{f}.jpg"
fft_label_file  = data_dir / f"{image_stem}_ftt_{f}.txt"
fft_visual_file = data_dir / f"{image_stem}_ftt_{f}_viz.jpg"

# Load data
image   = cv2.imread(str(image_file))
h, w, _ = image.shape
cropsz  = min(h, w)
bs      = mon.bbox.load(path=label_file, fmt=mon.BBoxFormat.YOLO, imgsz=(h, w))

classes = mon.load_config(classes_file, verbose=False)
classes = classes.get("classes", [])

# Preprocessing
# image, bs = mon.bbox.pad_square(image=image, bbox=bs)

# Transform
FFT = FisheyeTomographyTransform(f=f, imgsz=cropsz)
FFT.set_ext_params([0, 0, 0, 0, 0, 0])

transformed = FFT(image=image, bboxes=bs)
fft_image   = transformed["image"]
fft_bboxes  = transformed["bboxes"]

# Postprocessing
fft_image, fft_bboxes = mon.bbox.crop_fit_square(fft_image, fft_bboxes)

# Visualize
fft_bboxes_voc = mon.bbox.convert(fft_bboxes, fmt=mon.BBoxFormat.YOLO2VOC, imgsz=fft_image.shape[0:2])
fft_viz = fft_image.copy()
for j, b in enumerate(fft_bboxes_voc):
    if len(b) >= 6:
        l = f"{j} {int(b[5])}: {b[6]:.4f}"
    else:
        l = f"{j} {int(b[5])}"
    fft_viz = mon.dtypes.draw_bbox(
        image     = fft_viz,
        bbox      = b,
        label     = "",
        color     = classes[int(b[5])]["color"],
        thickness = 2,
        fill      = False,
    )

# Save
# cv2.imwrite(str(fft_image_file),  fft_image)
cv2.imwrite(str(fft_visual_file), fft_viz)

# with open(fft_label_file, "w") as f:
#     for b in fft_bboxes:
#         f.write(f"{int(b[5])} {b[0]:.32f} {b[1]:.32f} {b[2]:.32f} {b[3]:.32f}\n")